# Consumer Cellular - ML Financial Models

This notebook trains and evaluates three ML models for the CCI Intelligence platform:
1. **Churn Prediction Model** - Predicts customer churn probability
2. **Customer Lifetime Value (LTV) Model** - Predicts customer lifetime value
3. **Plan Recommendation Model** - Recommends optimal plans based on usage patterns

In [ ]:
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.snowpark.context import get_active_session

try:
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.getOrCreate()

In [ ]:
%%sql -r customer_features
SELECT
    c.CUSTOMER_ID,
    c.AGE,
    c.CUSTOMER_SEGMENT,
    c.AARP_MEMBER,
    DATEDIFF(MONTH, c.SIGNUP_DATE, CURRENT_DATE()) AS TENURE_MONTHS,
    s.STATUS AS SUBSCRIPTION_STATUS,
    s.AUTO_PAY_ENABLED,
    p.PLAN_TYPE,
    p.MONTHLY_PRICE,
    COALESCE(ltv.AVG_MONTHLY_REVENUE, 0) AS AVG_MONTHLY_REVENUE,
    COALESCE(ltv.HISTORICAL_REVENUE, 0) AS HISTORICAL_REVENUE,
    COALESCE(ticket_counts.TICKET_COUNT, 0) AS SUPPORT_TICKET_COUNT,
    COALESCE(late_payments.LATE_PAYMENT_COUNT, 0) AS LATE_PAYMENT_COUNT,
    COALESCE(churn.CHURN_PROBABILITY, 0.5) AS CHURN_PROBABILITY,
    CASE WHEN s.STATUS = 'Cancelled' THEN 1 ELSE 0 END AS IS_CHURNED
FROM CCI_INTELLIGENCE.RAW.CUSTOMERS c
LEFT JOIN CCI_INTELLIGENCE.RAW.SUBSCRIPTIONS s ON c.CUSTOMER_ID = s.CUSTOMER_ID
LEFT JOIN CCI_INTELLIGENCE.RAW.PLANS p ON s.PLAN_ID = p.PLAN_ID
LEFT JOIN CCI_INTELLIGENCE.RAW.CUSTOMER_LTV ltv ON c.CUSTOMER_ID = ltv.CUSTOMER_ID
LEFT JOIN CCI_INTELLIGENCE.RAW.CHURN_PREDICTIONS churn ON c.CUSTOMER_ID = churn.CUSTOMER_ID
LEFT JOIN (
    SELECT CUSTOMER_ID, COUNT(*) AS TICKET_COUNT
    FROM CCI_INTELLIGENCE.RAW.SUPPORT_TICKETS
    GROUP BY CUSTOMER_ID
) ticket_counts ON c.CUSTOMER_ID = ticket_counts.CUSTOMER_ID
LEFT JOIN (
    SELECT CUSTOMER_ID, COUNT(*) AS LATE_PAYMENT_COUNT
    FROM CCI_INTELLIGENCE.RAW.BILLING
    WHERE PAYMENT_STATUS = 'Past Due'
    GROUP BY CUSTOMER_ID
) late_payments ON c.CUSTOMER_ID = late_payments.CUSTOMER_ID

In [ ]:
print(f"Total records: {len(customer_features)}")
print(f"\nChurn distribution:")
print(customer_features['IS_CHURNED'].value_counts())
print(f"\nFeature columns: {list(customer_features.columns)}")

## Model 1: Churn Prediction

Binary classification model to predict whether a customer will churn.

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = customer_features.copy()

le_segment = LabelEncoder()
df['CUSTOMER_SEGMENT_ENC'] = le_segment.fit_transform(df['CUSTOMER_SEGMENT'].fillna('Unknown'))

le_plan = LabelEncoder()
df['PLAN_TYPE_ENC'] = le_plan.fit_transform(df['PLAN_TYPE'].fillna('Unknown'))

le_status = LabelEncoder()
df['SUB_STATUS_ENC'] = le_status.fit_transform(df['SUBSCRIPTION_STATUS'].fillna('Unknown'))

df['AARP_MEMBER'] = df['AARP_MEMBER'].astype(int)
df['AUTO_PAY_ENABLED'] = df['AUTO_PAY_ENABLED'].astype(int)

churn_features = ['AGE', 'TENURE_MONTHS', 'AARP_MEMBER', 'AUTO_PAY_ENABLED',
                  'MONTHLY_PRICE', 'AVG_MONTHLY_REVENUE', 'HISTORICAL_REVENUE',
                  'SUPPORT_TICKET_COUNT', 'LATE_PAYMENT_COUNT',
                  'CUSTOMER_SEGMENT_ENC', 'PLAN_TYPE_ENC']

X = df[churn_features].fillna(0)
y = df['IS_CHURNED']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

churn_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
churn_model.fit(X_train, y_train)

y_pred = churn_model.predict(X_test)
y_proba = churn_model.predict_proba(X_test)[:, 1]

print("=== Churn Prediction Model ===")
print(f"\nROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

feature_importance = pd.DataFrame({
    'feature': churn_features,
    'importance': churn_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop Feature Importances:")
print(feature_importance.to_string(index=False))

## Model 2: Customer Lifetime Value (LTV) Prediction

Regression model to predict customer lifetime value.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

ltv_target = df['HISTORICAL_REVENUE'].fillna(0)
ltv_features_cols = ['AGE', 'TENURE_MONTHS', 'AARP_MEMBER', 'AUTO_PAY_ENABLED',
                     'MONTHLY_PRICE', 'AVG_MONTHLY_REVENUE',
                     'SUPPORT_TICKET_COUNT', 'LATE_PAYMENT_COUNT',
                     'CUSTOMER_SEGMENT_ENC', 'PLAN_TYPE_ENC', 'SUB_STATUS_ENC']

X_ltv = df[ltv_features_cols].fillna(0)
y_ltv = ltv_target

X_ltv_train, X_ltv_test, y_ltv_train, y_ltv_test = train_test_split(X_ltv, y_ltv, test_size=0.2, random_state=42)

ltv_model = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
ltv_model.fit(X_ltv_train, y_ltv_train)

y_ltv_pred = ltv_model.predict(X_ltv_test)

print("=== LTV Prediction Model ===")
print(f"\nR² Score: {r2_score(y_ltv_test, y_ltv_pred):.4f}")
print(f"MAE: ${mean_absolute_error(y_ltv_test, y_ltv_pred):.2f}")
print(f"Mean Actual LTV: ${y_ltv_test.mean():.2f}")
print(f"Mean Predicted LTV: ${y_ltv_pred.mean():.2f}")

ltv_feature_importance = pd.DataFrame({
    'feature': ltv_features_cols,
    'importance': ltv_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop Feature Importances:")
print(ltv_feature_importance.to_string(index=False))

## Model 3: Plan Recommendation

Classification model to recommend the optimal plan based on customer usage patterns.

In [ ]:
%%sql -r usage_features
SELECT
    u.CUSTOMER_ID,
    AVG(u.DATA_USED_GB) AS AVG_DATA_GB,
    AVG(u.TALK_MINUTES_USED) AS AVG_TALK_MINUTES,
    AVG(u.TEXTS_SENT) AS AVG_TEXTS,
    SUM(u.OVERAGE_CHARGES) AS TOTAL_OVERAGE,
    p.PLAN_TYPE AS CURRENT_PLAN_TYPE,
    p.PLAN_NAME AS CURRENT_PLAN_NAME,
    p.DATA_LIMIT_GB,
    p.MONTHLY_PRICE
FROM CCI_INTELLIGENCE.RAW.USAGE_DATA u
JOIN CCI_INTELLIGENCE.RAW.SUBSCRIPTIONS s ON u.SUBSCRIPTION_ID = s.SUBSCRIPTION_ID
JOIN CCI_INTELLIGENCE.RAW.PLANS p ON s.PLAN_ID = p.PLAN_ID
GROUP BY u.CUSTOMER_ID, p.PLAN_TYPE, p.PLAN_NAME, p.DATA_LIMIT_GB, p.MONTHLY_PRICE

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

uf = usage_features.copy()

le_plan_rec = LabelEncoder()
uf['PLAN_ENCODED'] = le_plan_rec.fit_transform(uf['CURRENT_PLAN_TYPE'].fillna('Unknown'))

plan_rec_features = ['AVG_DATA_GB', 'AVG_TALK_MINUTES', 'AVG_TEXTS', 'TOTAL_OVERAGE',
                     'DATA_LIMIT_GB', 'MONTHLY_PRICE']

X_plan = uf[plan_rec_features].fillna(0)
y_plan = uf['PLAN_ENCODED']

X_plan_train, X_plan_test, y_plan_train, y_plan_test = train_test_split(
    X_plan, y_plan, test_size=0.2, random_state=42)

plan_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
plan_model.fit(X_plan_train, y_plan_train)

y_plan_pred = plan_model.predict(X_plan_test)

print("=== Plan Recommendation Model ===")
print(f"\nAccuracy: {accuracy_score(y_plan_test, y_plan_pred):.4f}")
print(f"\nPlan Types: {list(le_plan_rec.classes_)}")
print(f"\nClassification Report:")
print(classification_report(y_plan_test, y_plan_pred,
                            target_names=le_plan_rec.classes_,
                            zero_division=0))

plan_feature_importance = pd.DataFrame({
    'feature': plan_rec_features,
    'importance': plan_model.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(plan_feature_importance.to_string(index=False))

## Model Summary

All three models have been trained and evaluated. Summary of results below.

In [ ]:
print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"\n{'Model':<30} {'Key Metric':<20} {'Value'}")
print("-" * 60)
print(f"{'Churn Prediction':<30} {'ROC AUC':<20} {roc_auc_score(y_test, y_proba):.4f}")
print(f"{'LTV Prediction':<30} {'R² Score':<20} {r2_score(y_ltv_test, y_ltv_pred):.4f}")
print(f"{'Plan Recommendation':<30} {'Accuracy':<20} {accuracy_score(y_plan_test, y_plan_pred):.4f}")
print("=" * 60)